In [1]:
# ============ 셀 1: 설정 (L6A: SAM 구간 길이 축 + decay ρ 축) ============
import os, json, time, math, random, glob, shutil
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

assert torch.cuda.is_available(), "GPU가 꺼져 있어요. Settings → Accelerator → GPU T4 로 바꾸세요."
DEV = 'cuda'

OUT  = '/kaggle/working/exp_l6_pretrain'
DATA = '/tmp/llm_data'
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA, exist_ok=True)

CFG = dict(
    seed=0, ctx=256, vocab=50304,
    d_model=512, n_layer=6, n_head=8,
    # 사전학습: L5A와 완전히 동일
    pre_epochs=10, pre_bs=32, pre_wd=0.1, warmup_frac=0.02, decay_frac=0.2, grad_clip=1.0,
    muon_lr=0.005, adam_lr_aux=1e-3, muon_momentum=0.95, ns_steps=5,
    euc_rho=0.05, spec_rhos=[5e-4, 2e-3, 8e-3], ns_inner=5,
    # 평가·측정 (L5A와 동일)
    eval_seqs=400, eval_bs=32, hvp_iters=20,
    gap_euc_rho=0.05, gap_spec_rho=2e-3,
    # L6 신규: 체크포인트 지점
    ckpt_fracs=[0.4, 0.6, 0.8],
)
json.dump(CFG, open(f'{OUT}/config.json', 'w'), indent=2)

# (이름, 출발 체크포인트 비율, SAM 방식, ρ, SAM 구간 비율)
BRANCHES = [
    ('pre_off_repro',        0.8, 'none', 0.0,   0.0),
    ('pre_decay_spec_0.02',  0.8, 'spec', 0.02,  0.2),
    ('pre_decay_spec_0.05',  0.8, 'spec', 0.05,  0.2),
    ('pre_win40_spec_0.008', 0.6, 'spec', 0.008, 0.4),
    ('pre_win60_spec_0.008', 0.4, 'spec', 0.008, 0.6),
]

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print('GPU :', torch.cuda.get_device_name(0), '| torch', torch.__version__)

GPU : Tesla T4 | torch 2.11.0+cu128


In [2]:
# ============ 셀 2: BabyLM 다운로드, 토큰화 (L1·L2와 같은 방식) ============
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('gpt2'); tok.model_max_length = int(1e9)
EOS = tok.eos_token_id

def encode_texts(texts, bs=256):
    parts = []
    for i in range(0, len(texts), bs):
        for ids in tok(texts[i:i + bs])['input_ids']:
            parts.append(np.array(ids + [EOS], dtype=np.uint16))
    return np.concatenate(parts)

def chunk_lines(text, n=200):
    lines = text.split('\n')
    return ['\n'.join(lines[i:i + n]) for i in range(0, len(lines), n)]

if os.path.exists(f'{DATA}/babylm_val.npy'):
    print('BabyLM 토큰 파일이 이미 있음 → 건너뜀')
else:
    try:
        from huggingface_hub import snapshot_download
        root = snapshot_download('cambridge-climb/BabyLM', repo_type='dataset', allow_patterns=['10M/*', 'dev/*'])
        tr_files = sorted(f for f in glob.glob(f'{root}/10M/*') if os.path.isfile(f))
        dv_files = sorted(f for f in glob.glob(f'{root}/dev/*') if os.path.isfile(f))
        assert tr_files and dv_files, '파일 없음'
        tr_texts, dv_texts = [], []
        for f in tr_files: tr_texts += chunk_lines(open(f, encoding='utf-8', errors='ignore').read())
        for f in dv_files: dv_texts += chunk_lines(open(f, encoding='utf-8', errors='ignore').read()[:500_000])
        src = 'cambridge-climb/BabyLM (2023)'
    except Exception as e:
        print('2023판 다운로드 실패 → 2026판으로 대체:', repr(e)[:200])
        from datasets import load_dataset
        ds = load_dataset('BabyLM-community/BabyLM-2026-Strict-Small', split='train')
        col = [c for c, t in ds.features.items() if getattr(t, 'dtype', None) == 'string'][0]
        lines = [t for t in ds[col] if t and t.strip()]
        idx = np.random.RandomState(0).permutation(len(lines)); nv = len(lines) // 20
        dv_lines = [lines[i] for i in idx[:nv]]; tr_lines = [lines[i] for i in idx[nv:]]
        tr_texts = ['\n'.join(tr_lines[i:i + 200]) for i in range(0, len(tr_lines), 200)]
        dv_texts = ['\n'.join(dv_lines[i:i + 200]) for i in range(0, len(dv_lines), 200)]
        src = 'BabyLM-community/BabyLM-2026-Strict-Small (5%를 검증용으로 분리)'
    print('BabyLM 출처:', src, flush=True)
    tr = encode_texts(tr_texts); dv = encode_texts(dv_texts)
    np.save(f'{DATA}/babylm_train.npy', tr); np.save(f'{DATA}/babylm_val.npy', dv)
    print(f'BabyLM 토큰: train {len(tr)/1e6:.1f}M, val {len(dv)/1e6:.1f}M   (L1·L2: 15.7M / 0.8M)')

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

open_subtitles.train.txt: downloading bytes:           |  0.00B            

simple_wiki.train.txt:   0%|          | 0.00/8.86M [00:00<?, ?B/s]

switchboard.train.txt:   0%|          | 0.00/123k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1104106 [00:00<?, ? examples/s]

BabyLM 출처: BabyLM-community/BabyLM-2026-Strict-Small (5%를 검증용으로 분리)
BabyLM 토큰: train 15.7M, val 0.8M   (L1·L2: 15.7M / 0.8M)


In [3]:
# ============ 셀 3: 모델, Muon, SAM(유클리드/스펙트럼), 측정 도구 ============
def load_tokens(name):
    return torch.from_numpy(np.load(f'{DATA}/{name}.npy').astype(np.int64)).to(DEV)

TR_BABY, VA_BABY = load_tokens('babylm_train'), load_tokens('babylm_val')

class Stream:
    def __init__(self, tokens, bs, ctx=CFG['ctx']):
        n = (len(tokens) - 1) // ctx
        self.chunks = tokens[:n * ctx + 1].unfold(0, ctx + 1, ctx)
        self.n, self.bs, self.spe = n, bs, n // bs
        self._key = None
    def batch(self, step, seed):
        ep, i = divmod(step, self.spe)
        if self._key != (seed, ep):
            g = torch.Generator().manual_seed(seed * 100003 + ep)
            self._perm = torch.randperm(self.n, generator=g).to(DEV); self._key = (seed, ep)
        b = self.chunks[self._perm[i * self.bs:(i + 1) * self.bs]]
        return b[:, :-1], b[:, 1:]

def make_eval(tokens, n, seed, ctx=CFG['ctx']):
    starts = np.random.RandomState(seed).randint(0, len(tokens) - ctx - 1, size=n)
    idx = torch.from_numpy(starts).to(DEV)[:, None] + torch.arange(ctx + 1, device=DEV)
    b = tokens[idx]
    return b[:, :-1], b[:, 1:]

EV_BABY = make_eval(VA_BABY, CFG['eval_seqs'], seed=11)    # L1·L2·L3·L5와 같은 평가 문장

class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h = h
        self.ln1 = nn.LayerNorm(d); self.qkv = nn.Linear(d, 3 * d); self.proj = nn.Linear(d, d)
        self.ln2 = nn.LayerNorm(d); self.fc = nn.Linear(d, 4 * d); self.fc2 = nn.Linear(4 * d, d)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(self.ln1(x)).split(C, dim=2)
        q, k, v = [t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.proj(y.transpose(1, 2).contiguous().view(B, T, C))
        return x + self.fc2(F.gelu(self.fc(self.ln2(x))))

class GPT(nn.Module):
    def __init__(self, V=CFG['vocab'], d=CFG['d_model'], L=CFG['n_layer'], h=CFG['n_head'], T=CFG['ctx']):
        super().__init__()
        self.wte = nn.Embedding(V, d); self.wpe = nn.Embedding(T, d)
        self.blocks = nn.ModuleList([Block(d, h) for _ in range(L)])
        self.lnf = nn.LayerNorm(d)
        for n, p in self.named_parameters():
            if p.ndim >= 2:
                std = 0.02 / math.sqrt(2 * L) if n.endswith(('proj.weight', 'fc2.weight')) else 0.02
                nn.init.normal_(p, 0.0, std)
            elif n.endswith('bias'):
                nn.init.zeros_(p)
    def forward(self, idx, targets):
        x = self.wte(idx) + self.wpe(torch.arange(idx.shape[1], device=idx.device))
        for b in self.blocks:
            x = b(x)
        logits = F.linear(self.lnf(x), self.wte.weight)
        return F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))

def make_model():
    return GPT().to(DEV)

# ---------- Muon ----------
def newton_schulz5(G, steps):
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.float()
    X = X / (X.norm() + 1e-7)
    tr = X.size(0) > X.size(1)
    if tr: X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * (A @ A)
        X = a * X + B @ X
    return X.T if tr else X

class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, momentum=0.95, nesterov=True, ns_steps=5):
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov, ns_steps=ns_steps))
    @torch.no_grad()
    def step(self, closure=None):
        for grp in self.param_groups:
            for p in grp['params']:
                if p.grad is None: continue
                st = self.state[p]
                if 'buf' not in st: st['buf'] = torch.zeros_like(p.grad)
                buf = st['buf']; buf.mul_(grp['momentum']).add_(p.grad)
                u = p.grad.add(buf, alpha=grp['momentum']) if grp['nesterov'] else buf
                u = newton_schulz5(u, grp['ns_steps'])
                u *= max(1.0, p.size(0) / p.size(1)) ** 0.5
                p.add_(u.to(p.dtype), alpha=-grp['lr'])

def make_muon_opts(model, muon_lr, adam_lr, wd):
    hidden = [p for p in model.blocks.parameters() if p.ndim == 2]
    hid = {id(p) for p in hidden}
    rest = [p for p in model.parameters() if id(p) not in hid]
    muon = Muon(hidden, lr=muon_lr, momentum=CFG['muon_momentum'], ns_steps=CFG['ns_steps'])
    adam = torch.optim.AdamW([{'params': [p for p in rest if p.ndim >= 2], 'weight_decay': wd},
                              {'params': [p for p in rest if p.ndim < 2], 'weight_decay': 0.0}],
                             lr=adam_lr, betas=(0.9, 0.95), eps=1e-8)
    for o, lr in ((muon, muon_lr), (adam, adam_lr)):
        for g in o.param_groups: g['base_lr'] = lr
    return [muon, adam]

def new_opts(model):
    return make_muon_opts(model, CFG['muon_lr'], CFG['adam_lr_aux'], CFG['pre_wd'])

def wsd_fn(total, warm, decay_start):
    def f(s):
        if s < warm: return (s + 1) / warm
        if s < decay_start: return 1.0
        return max(0.0, (total - s) / (total - decay_start))
    return f

# ---------- SAM 흔들기 방향 ----------
@torch.no_grad()
def make_eps(params, mode, rho):
    """mode='euc': 모든 파라미터를 한 벡터로 보고 전체 유클리드 노름으로 정규화 (표준 SAM)
       mode='spec': 행렬마다 Newton-Schulz로 직교화한 방향 × rho (7월 논문 식 9), 행렬 아닌 파라미터는 0"""
    norms = torch.stack([p.grad.norm(2) for p in params])
    if not torch.isfinite(norms).all().item():
        return None
    if mode == 'euc':
        gn = norms.norm(2)
        if gn.item() == 0: return None
        k = rho / (gn + 1e-12)
        return [p.grad * k for p in params]
    if mode == 'spec':
        return [rho * newton_schulz5(p.grad, CFG['ns_inner']).to(p.dtype) if p.ndim == 2 else None for p in params]
    raise ValueError(mode)

def train_step(model, opts, scaler, x, y, mode, rho):
    """SAM Overlay: SAM은 흔드는 단계(ascent)만 담당, 업데이트는 Muon/AdamW가 각자 상태로 수행"""
    with torch.autocast('cuda', dtype=torch.float16):
        loss = model(x, y)
    model.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    used = 0
    if mode != 'none' and rho > 0:
        params = [p for p in model.parameters() if p.grad is not None]
        eps = make_eps(params, mode, rho)
        if eps is not None:
            with torch.no_grad():
                backup = [p.detach().clone() for p in params]
                for p, e in zip(params, eps):
                    if e is not None: p.add_(e)
            model.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                loss2 = model(x, y)
            scaler.scale(loss2).backward()
            with torch.no_grad():
                for p, b in zip(params, backup): p.copy_(b)
            used = 1
    for o in opts: scaler.unscale_(o)
    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
    for o in opts: scaler.step(o)
    scaler.update()
    return loss.detach(), used

@torch.no_grad()
def eval_loss(model, ev):
    model.eval(); X, Y = ev; tot = 0.0; bs = CFG['eval_bs']
    for i in range(0, len(X), bs):
        with torch.autocast('cuda', dtype=torch.float16):
            tot += model(X[i:i + bs], Y[i:i + bs]).float().item() * len(X[i:i + bs])
    model.train()
    return tot / len(X)

def train_steps(model, opts, scaler, s_from, s_to, lr_fn, mode, rho, log, eval_every, tag, total):
    model.train(); t0 = time.time(); acc = torch.zeros((), device=DEV); cnt = 0; used = 0
    for s in range(s_from, s_to):
        f = lr_fn(s)
        for o in opts:
            for g in o.param_groups: g['lr'] = g['base_lr'] * f
        x, y = tr_stream.batch(s, CFG['seed'])
        l, u = train_step(model, opts, scaler, x, y, mode, rho)
        acc += l; cnt += 1; used += u
        if (s + 1) % eval_every == 0 or s + 1 == s_to:
            vl = eval_loss(model, EV_BABY); tl = (acc / cnt).item(); dt = time.time() - t0
            log.append(dict(step=s + 1, train_loss=tl, val_loss=vl, sam_steps=used, sec=dt))
            print(f'[{tag}] step {s+1}/{total}  train {tl:.3f}  BabyLM val {vl:.4f} (ppl {math.exp(min(vl, 20)):.1f})  '
                  f'SAM {used}/{cnt}  {dt/cnt:.3f}s/step', flush=True)
            acc.zero_(); cnt = 0; used = 0; t0 = time.time()

# ---------- 평탄도 측정 ----------
def sharp_gap(model, batches, mode, rho):
    model.eval(); gaps = []; params = list(model.parameters())
    for x, y in batches:
        model.zero_grad(set_to_none=True)
        loss = model(x, y); loss.backward()
        eps = make_eps(params, mode, rho)
        with torch.no_grad():
            backup = [p.detach().clone() for p in params]
            for p, e in zip(params, eps):
                if e is not None: p.add_(e)
            l2 = model(x, y)
            for p, b in zip(params, backup): p.copy_(b)
        gaps.append((l2 - loss).item())
    model.zero_grad(set_to_none=True)
    return float(np.mean(gaps))

def lambda_max(model, batch, iters):
    model.eval(); x, y = batch; params = list(model.parameters())
    with sdpa_kernel(SDPBackend.MATH):
        loss = model(x, y)
        grads = torch.autograd.grad(loss, params, create_graph=True)
        v = [torch.randn_like(p) for p in params]
        nrm = torch.sqrt(sum((a * a).sum() for a in v)); v = [a / nrm for a in v]
        lam = 0.0
        for _ in range(iters):
            Hv = torch.autograd.grad(grads, params, grad_outputs=v, retain_graph=True)
            lam = sum((h * a).sum() for h, a in zip(Hv, v)).item()
            nrm = torch.sqrt(sum((h * h).sum() for h in Hv)); v = [h.detach() / nrm for h in Hv]
    del grads
    return lam

tr_stream = Stream(TR_BABY, CFG['pre_bs'])
TOTAL = CFG['pre_epochs'] * tr_stream.spe
S_dec = int(TOTAL * (1 - CFG['decay_frac']))
lr_fn = wsd_fn(TOTAL, max(1, int(TOTAL * CFG['warmup_frac'])), S_dec)
EVAL_EVERY = max(1, TOTAL // 20)

log_path, time_path = f'{OUT}/pretrain_log.json', f'{OUT}/timing.json'
LOG = json.load(open(log_path)) if os.path.exists(log_path) else {}
TIME = json.load(open(time_path)) if os.path.exists(time_path) else {}
def save_logs():
    json.dump(LOG, open(log_path, 'w'), indent=1); json.dump(TIME, open(time_path, 'w'), indent=1)

print(f'epoch당 {tr_stream.spe} step, 전체 {TOTAL} step, 감쇠 시작 {S_dec} step (L2: 1914 / 19140 / 15312)')

epoch당 1914 step, 전체 19140 step, 감쇠 시작 15312 step (L2: 1914 / 19140 / 15312)


In [4]:
# ============ 셀 4: 공통 구간 (SAM 없음) — 40% / 60% / 80% 지점 체크포인트 저장 ============
CK_STEPS = {f: int(round(TOTAL * f)) for f in CFG['ckpt_fracs']}
assert all(s % EVAL_EVERY == 0 for s in CK_STEPS.values()), CK_STEPS
print('체크포인트 지점:', CK_STEPS)

def ck_path(f): return f'{OUT}/pre_shared_{int(f*100)}.pt'
resume = f'{OUT}/shared_resume.pt'

seed_all(CFG['seed'])
model = make_model(); opts = new_opts(model); scaler = torch.amp.GradScaler('cuda')
start, sec = 0, 0.0
if os.path.exists(resume):
    ck = torch.load(resume, map_location=DEV)
    model.load_state_dict(ck['model']); scaler.load_state_dict(ck['scaler'])
    for o, sd in zip(opts, ck['opts']): o.load_state_dict(sd)
    start, sec = ck['step'], ck['sec']; print(f'체크포인트 발견 → step {start}부터 이어서')
LOG.setdefault('shared', [])
s = start
while s < S_dec:
    e = min(S_dec, s + EVAL_EVERY); t0 = time.time()
    train_steps(model, opts, scaler, s, e, lr_fn, 'none', 0.0, LOG['shared'], EVAL_EVERY, 'shared', TOTAL)
    sec += time.time() - t0; s = e
    state = dict(model=model.state_dict(), opts=[o.state_dict() for o in opts], scaler=scaler.state_dict(), step=s, sec=sec)
    torch.save(state, resume)
    for f, st in CK_STEPS.items():
        if s == st:
            torch.save(state, ck_path(f)); TIME[f'shared_{int(f*100)}'] = sec
            print(f'  → {int(f*100)}% 체크포인트 저장 (누적 {sec/60:.1f}분)')
    save_logs()
del model, opts; torch.cuda.empty_cache()
print('\n공통 구간 완료:', {k: f'{v/60:.1f}분' for k, v in TIME.items() if k.startswith('shared_')})

체크포인트 지점: {0.4: 7656, 0.6: 11484, 0.8: 15312}
[shared] step 957/19140  train 4.587  BabyLM val 3.8118 (ppl 45.2)  SAM 0/957  0.276s/step
[shared] step 1914/19140  train 3.695  BabyLM val 3.6196 (ppl 37.3)  SAM 0/957  0.299s/step
[shared] step 2871/19140  train 3.509  BabyLM val 3.5433 (ppl 34.6)  SAM 0/957  0.305s/step
[shared] step 3828/19140  train 3.476  BabyLM val 3.4875 (ppl 32.7)  SAM 0/957  0.305s/step
[shared] step 4785/19140  train 3.343  BabyLM val 3.4611 (ppl 31.9)  SAM 0/957  0.305s/step
[shared] step 5742/19140  train 3.372  BabyLM val 3.4317 (ppl 30.9)  SAM 0/957  0.305s/step
[shared] step 6699/19140  train 3.250  BabyLM val 3.4307 (ppl 30.9)  SAM 0/957  0.305s/step
[shared] step 7656/19140  train 3.296  BabyLM val 3.4065 (ppl 30.2)  SAM 0/957  0.305s/step
  → 40% 체크포인트 저장 (누적 38.4분)
[shared] step 8613/19140  train 3.176  BabyLM val 3.4190 (ppl 30.5)  SAM 0/957  0.305s/step
[shared] step 9570/19140  train 3.246  BabyLM val 3.4039 (ppl 30.1)  SAM 0/957  0.305s/step
[shared

In [5]:
# ============ 셀 5: 갈래 5개 (재현 확인 / decay ρ 0.02·0.05 / SAM 구간 40%·60%) ============
for tag, frac, mode, rho, win in BRANCHES:
    path = f'{OUT}/{tag}.pt'
    if os.path.exists(path):
        print(tag, '이미 있음 → 건너뜀'); continue
    ck = torch.load(ck_path(frac), map_location=DEV)
    assert ck['step'] == CK_STEPS[frac]
    model = make_model(); model.load_state_dict(ck['model'])
    opts = new_opts(model)
    for o, sd in zip(opts, ck['opts']): o.load_state_dict(sd)
    scaler = torch.amp.GradScaler('cuda'); scaler.load_state_dict(ck['scaler'])
    LOG[tag] = []
    print(f'\n=== 갈래 {tag}: step {ck["step"]+1}~{TOTAL}, SAM={mode}, rho={rho:g} ===')
    t0 = time.time()
    train_steps(model, opts, scaler, ck['step'], TOTAL, lr_fn, mode, rho, LOG[tag], EVAL_EVERY, tag, TOTAL)
    TIME[tag] = time.time() - t0
    vl = LOG[tag][-1]['val_loss']
    if not math.isfinite(vl):
        print(f'※ {tag} 발산 (loss {vl}) — 결과표에 표시만 하고 넘어감')
    torch.save(model.state_dict(), path); save_logs()
    print(f'{tag}: 최종 BabyLM loss {vl:.4f} | 갈래 {TIME[tag]/60:.1f}분')
    del model, opts, ck; torch.cuda.empty_cache()
print('\n갈래 완료')


=== 갈래 pre_off_repro: step 15313~19140, SAM=none, rho=0 ===
[pre_off_repro] step 16269/19140  train 2.963  BabyLM val 3.4152 (ppl 30.4)  SAM 0/957  0.305s/step
[pre_off_repro] step 17226/19140  train 2.987  BabyLM val 3.3612 (ppl 28.8)  SAM 0/957  0.305s/step
[pre_off_repro] step 18183/19140  train 2.710  BabyLM val 3.4071 (ppl 30.2)  SAM 0/957  0.305s/step
[pre_off_repro] step 19140/19140  train 2.707  BabyLM val 3.3800 (ppl 29.4)  SAM 0/957  0.305s/step
pre_off_repro: 최종 BabyLM loss 3.3800 | 갈래 19.5분

=== 갈래 pre_decay_spec_0.02: step 15313~19140, SAM=spec, rho=0.02 ===
[pre_decay_spec_0.02] step 16269/19140  train 3.018  BabyLM val 3.3222 (ppl 27.7)  SAM 957/957  0.690s/step
[pre_decay_spec_0.02] step 17226/19140  train 3.043  BabyLM val 3.2726 (ppl 26.4)  SAM 957/957  0.703s/step
[pre_decay_spec_0.02] step 18183/19140  train 2.847  BabyLM val 3.2535 (ppl 25.9)  SAM 957/957  0.690s/step
[pre_decay_spec_0.02] step 19140/19140  train 2.845  BabyLM val 3.2299 (ppl 25.3)  SAM 957/957  0

In [6]:
# ============ 셀 6: 평탄도 측정 (L5A와 같은 배치·같은 방법) ============
fixed = [tr_stream.batch(s, seed=999) for s in range(4)]
fixed = [(x[:16], y[:16]) for x, y in fixed]
PRE = {}
for tag, *_ in BRANCHES:
    m = make_model(); m.load_state_dict(torch.load(f'{OUT}/{tag}.pt', map_location=DEV))
    vl = eval_loss(m, EV_BABY)
    if not math.isfinite(vl):
        PRE[tag] = dict(babylm_loss=vl); print(f'{tag}: 발산'); continue
    ge = sharp_gap(m, fixed, 'euc', CFG['gap_euc_rho'])
    gs = sharp_gap(m, fixed, 'spec', CFG['gap_spec_rho'])
    lam = lambda_max(m, fixed[0], CFG['hvp_iters'])
    PRE[tag] = dict(babylm_loss=vl, gap_euc=ge, gap_spec=gs, lambda_max=lam)
    print(f'{tag:24s} BabyLM {vl:.4f} (ppl {math.exp(vl):.1f}) | 유클리드 gap {ge:.4f} | 스펙트럼 gap {gs:.4f} | lambda_max {lam:.1f}', flush=True)
    del m; torch.cuda.empty_cache()
json.dump(PRE, open(f'{OUT}/pretrain_check.json', 'w'), indent=2)

pre_off_repro            BabyLM 3.3800 (ppl 29.4) | 유클리드 gap 0.0730 | 스펙트럼 gap 0.0921 | lambda_max 127.3
pre_decay_spec_0.02      BabyLM 3.2299 (ppl 25.3) | 유클리드 gap 0.0488 | 스펙트럼 gap 0.0488 | lambda_max 57.3
pre_decay_spec_0.05      BabyLM 3.2429 (ppl 25.6) | 유클리드 gap 0.0404 | 스펙트럼 gap 0.0300 | lambda_max 39.5
pre_win40_spec_0.008     BabyLM 3.2694 (ppl 26.3) | 유클리드 gap 0.0535 | 스펙트럼 gap 0.0614 | lambda_max 89.8
pre_win60_spec_0.008     BabyLM 3.2664 (ppl 26.2) | 유클리드 gap 0.0501 | 스펙트럼 gap 0.0589 | lambda_max 82.2


In [7]:
# ============ 셀 7: 결과 정리 — L5A 결과와 합쳐 SAM 구간 길이 곡선 ============
import pandas as pd
rows = []
for tag, frac, mode, rho, win in BRANCHES:
    sec = TIME[f'shared_{int(frac*100)}'] + TIME[tag]
    rows.append(dict(model=tag, SAM구간_비율=win, 방식=mode, rho=rho, 사전학습_시간_분=sec / 60, **PRE[tag]))
df = pd.DataFrame(rows)
base = df.loc[df.model == 'pre_off_repro', '사전학습_시간_분'].item()
df['비용_배수'] = df['사전학습_시간_분'] / base
df.to_csv(f'{OUT}/pretrain_summary.csv', index=False)
display(df.round(4))

print('\n===== 재현 확인 =====')
r = PRE['pre_off_repro']['babylm_loss']
print(f'  pre_off_repro {r:.4f}  vs  L5A pre_off 3.3896   (차이 {r-3.3896:+.4f}, 0.003 이내면 재현 성공)')

print('\n===== SAM 구간 길이 축 (스펙트럼 ρ 0.008, 사전학습 BabyLM loss) =====')
L5A_REF = {0.0: 3.3896, 0.2: 3.2844, 1.0: 3.2648}
curve = dict(L5A_REF)
curve[0.4] = PRE['pre_win40_spec_0.008']['babylm_loss']
curve[0.6] = PRE['pre_win60_spec_0.008']['babylm_loss']
full_gain = L5A_REF[0.0] - L5A_REF[1.0]
for w in sorted(curve):
    gain = L5A_REF[0.0] - curve[w]
    src = 'L5A' if w in L5A_REF else 'L6A'
    print(f'  SAM {int(w*100):3d}%: loss {curve[w]:.4f} | 개선 {gain:.4f} = full의 {gain/full_gain*100:5.1f}%  ({src})')

print('\n===== decay 20% 구간의 ρ 축 (사전학습 BabyLM loss) =====')
for rho, v in [(5e-4, 3.3787), (2e-3, 3.3523), (8e-3, 3.2844),
               (0.02, PRE['pre_decay_spec_0.02']['babylm_loss']), (0.05, PRE['pre_decay_spec_0.05']['babylm_loss'])]:
    print(f'  ρ {rho:<6g}: {v:.4f}  ({"L5A" if rho < 0.01 else "L6A"})')

print(f'\n파인튜닝(L6B)에 쓸 모델 파일: {OUT}/' + ', '.join(f'{t}.pt' for t, *_ in BRANCHES))

,model,SAM구간_비율,방식,rho,사전학습_시간_분,babylm_loss,gap_euc,gap_spec,lambda_max,비용_배수
0,pre_off_repro,0.0,none,0.000,96.9004,3.3800,0.0730,0.0921,127.3366,1.0000
1,pre_decay_spec_0.02,0.2,spec,0.020,121.6716,3.2299,0.0488,0.0488,57.3415,1.2556
2,pre_decay_spec_0.05,0.2,spec,0.050,121.7253,3.2429,0.0404,0.0300,39.5069,1.2562
3,pre_win40_spec_0.008,0.4,spec,0.008,146.3754,3.2694,0.0535,0.0614,89.7765,1.5106
4,pre_win60_spec_0.008,0.6,spec,0.008,171.1767,3.2664,0.0501,0.0589,82.1706,1.7665



===== 재현 확인 =====
  pre_off_repro 3.3800  vs  L5A pre_off 3.3896   (차이 -0.0096, 0.003 이내면 재현 성공)

===== SAM 구간 길이 축 (스펙트럼 ρ 0.008, 사전학습 BabyLM loss) =====
  SAM   0%: loss 3.3896 | 개선 0.0000 = full의   0.0%  (L5A)
  SAM  20%: loss 3.2844 | 개선 0.1052 = full의  84.3%  (L5A)
  SAM  40%: loss 3.2694 | 개선 0.1202 = full의  96.4%  (L6A)
  SAM  60%: loss 3.2664 | 개선 0.1232 = full의  98.7%  (L6A)
  SAM 100%: loss 3.2648 | 개선 0.1248 = full의 100.0%  (L5A)

===== decay 20% 구간의 ρ 축 (사전학습 BabyLM loss) =====
  ρ 0.0005: 3.3787  (L5A)
  ρ 0.002 : 3.3523  (L5A)
  ρ 0.008 : 3.2844  (L5A)
  ρ 0.02  : 3.2299  (L6A)
  ρ 0.05  : 3.2429  (L6A)

파인튜닝(L6B)에 쓸 모델 파일: /kaggle/working/exp_l6_pretrain/pre_off_repro.pt, pre_decay_spec_0.02.pt, pre_decay_spec_0.05.pt, pre_win40_spec_0.008.pt, pre_win60_spec_0.008.pt
